# Data wrangling — landing outcome
The official IBM lab typically uses a historical, fixed SpaceX dataset. This notebook attempts to retrieve it; if unavailable, falls back to the API dataset with a clearly documented different cohort.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
DATA=Path('../data');DATA.mkdir(exist_ok=True)
urls=['https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv']
df=None
for u in urls:
    try:
        df=pd.read_csv(u);print('Official course data:',u,df.shape);break
    except Exception as e: print('Download unavailable:',str(e)[:120])
if df is None:
    df=pd.read_csv(DATA/'spacex_api.csv');print('Using current API data, not original IBM course cohort')
display(df.head());print(df.columns.tolist())

In [ ]:
# Binary label: 1 = successful first-stage landing, 0 = other recorded outcomes.
# IMPORTANT: distinguish unknown/unreported from failure; do not silently turn NaN into 0.
if 'Class' in df:
    df['Class']=pd.to_numeric(df['Class'],errors='coerce')
elif 'Outcome' in df:
    # IBM outcomes commonly encode True/False landing status as text (e.g. True ASDS).
    outcome=df['Outcome'].astype('string')
    df['Class']=np.where(outcome.str.startswith('True',na=False),1,
                    np.where(outcome.str.startswith('False',na=False),0,np.nan))
elif 'LandingOutcome' in df:
    outcome=df['LandingOutcome'].astype('string').str.lower()
    df['Class']=outcome.map({'true':1,'false':0})
else: raise KeyError('No landing outcome column found')
print('Known outcome counts:');print(df['Class'].value_counts(dropna=False))
clean=df.dropna(subset=['Class']).copy();clean['Class']=clean['Class'].astype(int)
clean.to_csv(DATA/'dataset_part_2.csv',index=False)
print('Labeled data:',clean.shape);display(clean.head())